# AI Mathematical Olympiad — Pipeline Development

**Base model:** Qwen2.5-Math-7B

## Phases
1. **Setup** — install deps, verify GPU, download model
2. **Data Pipeline** — stream OpenMathReasoning TIR → structured parquet
3. **Data Understanding** — inspect 50 problems, categorize by type, measure structure
4. **Reality Check** — can SymPy actually verify reasoning steps? Measure success rate, document failure modes

**Before running:** Runtime → Change runtime type → **A100 GPU** (or T4 minimum)

---
## 1. Setup & GPU Verification

In [ ]:
!pip install -q vllm transformers>=4.44.0 datasets sympy pandas pyarrow tqdm

In [ ]:
import torch
import os

if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Go to Runtime -> Change runtime type -> GPU")

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"GPU:  {GPU_NAME}")
print(f"VRAM: {VRAM_GB:.1f} GB")
print(f"PyTorch: {torch.__version__}")

MODEL_ID = "Qwen/Qwen2.5-Math-7B"

# Directories
os.makedirs("/content/data/processed", exist_ok=True)
os.makedirs("/content/models", exist_ok=True)
os.makedirs("/content/experiments", exist_ok=True)

print(f"Base model: {MODEL_ID}")

---
## 2. Data Pipeline — Stream OpenMathReasoning TIR

Streams from `nvidia/OpenMathReasoning` (TIR split), parses each solution into
structured reasoning steps (text segments, code blocks, outputs, boxed answers),
and writes to a parquet file. Only the current batch is in memory.

**Output schema per row:**
- `problem_id` — unique tracking ID (`tir_000001`, ...)
- `problem` / `solution` / `expected_answer` / `difficulty` — raw fields
- `messages` — chat-format for `tokenizer.apply_chat_template()`
- `parsed_answer` — extracted from `\boxed{}`
- `reasoning_steps` — list of `{step_number, type, content}` dicts
- `num_steps` / `num_code_blocks` / `has_answer` — summary stats

In [ ]:
import gc
import re
import pyarrow as pa
import pyarrow.parquet as pq
from datasets import load_dataset
from tqdm.auto import tqdm

DATA_LIMIT = 50_000  # Set to None for full 1.7M rows
BATCH_SIZE = 5_000
OUTPUT_PATH = "/content/data/processed/math_reasoning_tir.parquet"

# --- Structured schema ---
STEP_STRUCT = pa.struct([
    ("step_number", pa.int32()),
    ("type", pa.string()),       # "text", "code", "output", "answer"
    ("content", pa.string()),
])

SCHEMA = pa.schema([
    ("problem_id", pa.string()),
    ("problem", pa.string()),
    ("solution", pa.string()),
    ("expected_answer", pa.string()),
    ("difficulty", pa.string()),
    ("messages", pa.list_(pa.struct([
        ("role", pa.string()),
        ("content", pa.string()),
    ]))),
    ("parsed_answer", pa.string()),
    ("reasoning_steps", pa.list_(STEP_STRUCT)),
    ("num_steps", pa.int32()),
    ("num_code_blocks", pa.int32()),
    ("has_answer", pa.bool_()),
])

_row_counter = 0


# --- Solution parser ---

def parse_solution(solution_text):
    """Parse a TIR solution into structured reasoning steps.

    TIR format alternates between:
      - free text (reasoning/explanation)
      - ```python ... ``` code blocks
      - ```output ... ``` execution results
      - \\boxed{...} final answers

    Returns (steps, parsed_answer) where steps is a list of dicts.
    """
    if not solution_text:
        return [], None

    steps = []
    step_num = 0
    parsed_answer = None

    # Split on code fences while keeping delimiters
    # Pattern captures: ```python\n...\n``` and ```output\n...\n```
    parts = re.split(r'(```(?:python|output)\n.*?```)', solution_text, flags=re.DOTALL)

    for part in parts:
        part = part.strip()
        if not part:
            continue

        if part.startswith('```python\n') and part.endswith('```'):
            step_num += 1
            code = part[len('```python\n'):-len('```')].strip()
            steps.append({"step_number": step_num, "type": "code", "content": code})

        elif part.startswith('```output\n') and part.endswith('```'):
            step_num += 1
            output = part[len('```output\n'):-len('```')].strip()
            steps.append({"step_number": step_num, "type": "output", "content": output})

        else:
            # Text segment — may contain \boxed{} answer
            boxed = re.findall(r'\\boxed\{(.+?)\}', part)
            if boxed:
                parsed_answer = boxed[-1]  # last boxed is the final answer

            # Split text on blank lines into logical chunks
            chunks = re.split(r'\n\s*\n', part)
            for chunk in chunks:
                chunk = chunk.strip()
                if not chunk:
                    continue
                step_num += 1
                # Tag chunks that contain a boxed answer
                step_type = "answer" if re.search(r'\\boxed\{', chunk) else "text"
                steps.append({"step_number": step_num, "type": step_type, "content": chunk})

    return steps, parsed_answer


def build_tir_messages(problem, solution):
    return [
        {"role": "system", "content": ""},
        {"role": "user", "content": problem},
        {"role": "assistant", "content": solution},
    ]


def format_tir_chat(problem, solution):
    return (
        f"<|system|>\n<|end|>\n"
        f"<|user|>\n{problem}<|end|>\n"
        f"<|assistant|>\n{solution}<|end|>"
    )


def transform_row(example):
    global _row_counter
    _row_counter += 1
    problem_id = f"tir_{_row_counter:06d}"
    problem = example["problem"]
    solution = example["generated_solution"]

    steps, parsed_answer = parse_solution(solution)
    num_code = sum(1 for s in steps if s["type"] == "code")

    return {
        "problem_id": problem_id,
        "problem": problem,
        "solution": solution,
        "expected_answer": example["expected_answer"],
        "difficulty": example.get("problem_source", "unknown"),
        "messages": build_tir_messages(problem, solution),
        "parsed_answer": parsed_answer,
        "reasoning_steps": steps,
        "num_steps": len(steps),
        "num_code_blocks": num_code,
        "has_answer": parsed_answer is not None,
    }


print("Streaming nvidia/OpenMathReasoning (tir split)...")
ds = load_dataset("nvidia/OpenMathReasoning", split="tir", streaming=True)

writer = None
batch = []
rows_written = 0

for example in tqdm(ds, desc="Processing", total=DATA_LIMIT):
    if DATA_LIMIT and rows_written + len(batch) >= DATA_LIMIT:
        break
    batch.append(transform_row(example))

    if len(batch) >= BATCH_SIZE:
        table = pa.Table.from_pylist(batch, schema=SCHEMA)
        if writer is None:
            writer = pq.ParquetWriter(OUTPUT_PATH, SCHEMA, compression='snappy')
        writer.write_table(table)
        rows_written += len(batch)
        batch.clear()
        del table
        gc.collect()
        print(f"  Written {rows_written:,} rows")

if batch:
    table = pa.Table.from_pylist(batch, schema=SCHEMA)
    if writer is None:
        writer = pq.ParquetWriter(OUTPUT_PATH, SCHEMA, compression='snappy')
    writer.write_table(table)
    rows_written += len(batch)
    batch.clear()
    del table

if writer:
    writer.close()

del ds, writer
gc.collect()

final_count = pq.read_metadata(OUTPUT_PATH).num_rows
print(f"\nDone: {final_count:,} rows -> {OUTPUT_PATH}")

In [ ]:
# Quick peek at the structured data
import pandas as pd

df_peek = pd.read_parquet(OUTPUT_PATH)
print(f"Total rows: {len(df_peek):,}")
print(f"Has parsed answer: {df_peek['has_answer'].sum():,} / {len(df_peek):,} ({df_peek['has_answer'].mean():.1%})")
print(f"Avg steps per solution: {df_peek['num_steps'].mean():.1f}")
print(f"Avg code blocks per solution: {df_peek['num_code_blocks'].mean():.1f}")

print(f"\nDifficulty distribution:")
print(df_peek["difficulty"].value_counts().head(10))

# Show one parsed example
row = df_peek.iloc[0]
print(f"\n{'='*60}")
print(f"problem_id: {row['problem_id']}")
print(f"Problem: {row['problem'][:200]}...")
print(f"Expected: {row['expected_answer']}")
print(f"Parsed answer: {row['parsed_answer']}")
print(f"Steps ({row['num_steps']} total, {row['num_code_blocks']} code blocks):")
for step in row["reasoning_steps"][:8]:
    preview = step["content"][:100].replace("\n", " ")
    print(f"  [{step['step_number']}] ({step['type']}) {preview}...")

del df_peek
gc.collect()

---
## 3. Data Understanding

Inspect 50 problems manually. Categorize by math type (algebra, number theory, geometry, combinatorics).
Measure structural properties: how many have code, how many have extractable equations, step counts by category.

In [ ]:
import re
import pandas as pd
import sympy

INSPECT_N = 50

df = pd.read_parquet(OUTPUT_PATH).head(INSPECT_N)

# --- Keyword-based category detection ---
CATEGORY_PATTERNS = {
    "algebra": re.compile(
        r'equation|solve\s+for|polynomial|factor|quadratic|linear|inequality|'
        r'simplif|variable|expression|roots?|coefficient|system\s+of', re.I
    ),
    "number_theory": re.compile(
        r'integer|prime|divisib|modulo|mod\s|gcd|lcm|remainder|divisor|'
        r'congruent|digit|even|odd|perfect\s+square|factorial', re.I
    ),
    "geometry": re.compile(
        r'triangle|circle|angle|perpendicular|parallel|area|perimeter|'
        r'radius|diameter|polygon|midpoint|bisect|tangent|chord|inscribed', re.I
    ),
    "combinatorics": re.compile(
        r'how\s+many\s+ways|combin|permut|choose|arrange|count|probability|'
        r'subset|sequence|binomial|pigeonhole|grid|path', re.I
    ),
}


def categorize_problem(problem_text):
    """Assign categories based on keyword matches. A problem can have multiple."""
    cats = []
    for cat, pat in CATEGORY_PATTERNS.items():
        if pat.search(problem_text):
            cats.append(cat)
    return cats if cats else ["other"]


def extract_equations(text):
    """Extract mathematical expressions from text that SymPy might parse."""
    eqs = []
    # LaTeX inline math: $...$
    eqs.extend(re.findall(r'\$([^$]+)\$', text))
    # Explicit equations with =
    eqs.extend(re.findall(r'([a-zA-Z0-9\s\+\-\*/\^()]+=[a-zA-Z0-9\s\+\-\*/\^()]+)', text))
    # \boxed{} expressions
    eqs.extend(re.findall(r'\\boxed\{(.+?)\}', text))
    return eqs


# Categorize all 50 problems
rows = []
for _, row in df.iterrows():
    cats = categorize_problem(row["problem"])
    eqs = extract_equations(row["solution"])
    rows.append({
        "problem_id": row["problem_id"],
        "categories": cats,
        "primary_category": cats[0],
        "num_equations": len(eqs),
        "num_steps": row["num_steps"],
        "num_code_blocks": row["num_code_blocks"],
        "has_answer": row["has_answer"],
        "problem_preview": row["problem"][:120],
    })

cat_df = pd.DataFrame(rows)

# Explode categories for counting (one problem can be in multiple)
exploded = cat_df.explode("categories")

print(f"=== Category Distribution (n={INSPECT_N}) ===")
print(exploded["categories"].value_counts().to_string())

print(f"\n=== Stats by Primary Category ===")
grouped = cat_df.groupby("primary_category").agg(
    count=("problem_id", "count"),
    avg_steps=("num_steps", "mean"),
    avg_code_blocks=("num_code_blocks", "mean"),
    avg_equations=("num_equations", "mean"),
    pct_has_answer=("has_answer", "mean"),
).round(2)
print(grouped.to_string())

print(f"\n=== Overall ===")
print(f"Problems with equations: {sum(1 for r in rows if r['num_equations'] > 0)}/{INSPECT_N}")
print(f"Problems with code blocks: {sum(1 for r in rows if r['num_code_blocks'] > 0)}/{INSPECT_N}")
print(f"Problems with parsed answer: {sum(1 for r in rows if r['has_answer'])}/{INSPECT_N}")

In [ ]:
# Show 5 sample problems per category for manual inspection
for cat in ["algebra", "number_theory", "geometry", "combinatorics", "other"]:
    subset = cat_df[cat_df["primary_category"] == cat].head(5)
    if len(subset) == 0:
        continue
    print(f"\n{'='*60}")
    print(f"  {cat.upper()} — {len(cat_df[cat_df['primary_category']==cat])} problems")
    print(f"{'='*60}")
    for _, r in subset.iterrows():
        print(f"\n[{r['problem_id']}] steps={r['num_steps']} code={r['num_code_blocks']} eqs={r['num_equations']}")
        print(f"  {r['problem_preview']}...")

---
## 4. Reality Check — Can SymPy Verify Reasoning Steps?

Take 20 problems with step-by-step solutions. Extract mathematical expressions from each step.
Try to verify each with SymPy. Measure success rate per category.

**Decision point:** If <60% of steps are verifiable, we need Z3 or must focus on algebra-heavy problems only.

In [ ]:
import sympy
from sympy.parsing.latex import parse_latex
from collections import defaultdict

VERIFY_N = 20

verify_df = pd.read_parquet(OUTPUT_PATH).head(VERIFY_N)


def try_sympy_parse(expr_str):
    """Try to parse a math expression with SymPy. Returns (parsed_expr, method, error)."""
    expr_str = expr_str.strip()
    if not expr_str:
        return None, None, "empty"

    # Try direct sympify first (handles plain math like "x**2 + 3")
    try:
        expr = sympy.sympify(expr_str)
        return expr, "sympify", None
    except Exception:
        pass

    # Try LaTeX parsing (handles \frac, \sqrt, etc.)
    try:
        expr = parse_latex(expr_str)
        return expr, "latex", None
    except Exception:
        pass

    # Try cleaning LaTeX → plain and sympify again
    cleaned = expr_str
    for old, new in [("\\cdot", "*"), ("\\times", "*"), ("\\div", "/"),
                     ("\\left", ""), ("\\right", ""), ("\\,", ""),
                     ("^", "**"), ("{", "("), ("}", ")")]:
        cleaned = cleaned.replace(old, new)
    cleaned = re.sub(r'\\frac\(([^)]+)\)\(([^)]+)\)', r'(\1)/(\2)', cleaned)
    cleaned = re.sub(r'\\sqrt\(([^)]+)\)', r'sqrt(\1)', cleaned)
    try:
        expr = sympy.sympify(cleaned)
        return expr, "cleaned", None
    except Exception as e:
        return None, None, str(e)


def try_verify_equation(eq_str):
    """Try to verify an equation (with =) using SymPy.
    Returns (success, method, detail)."""
    if "=" not in eq_str or "==" in eq_str or "!=" in eq_str or "\\neq" in eq_str:
        return None, None, "not_equation"

    parts = eq_str.split("=", 1)
    if len(parts) != 2:
        return None, None, "bad_split"

    lhs_str, rhs_str = parts[0].strip(), parts[1].strip()
    if not lhs_str or not rhs_str:
        return None, None, "empty_side"

    lhs, lhs_method, lhs_err = try_sympy_parse(lhs_str)
    rhs, rhs_method, rhs_err = try_sympy_parse(rhs_str)

    if lhs is None or rhs is None:
        return False, None, f"parse_fail: lhs={lhs_err}, rhs={rhs_err}"

    try:
        diff = sympy.simplify(lhs - rhs)
        if diff == 0:
            return True, f"{lhs_method}+{rhs_method}", "verified_equal"
        elif diff.is_number:
            return False, f"{lhs_method}+{rhs_method}", f"not_equal: diff={diff}"
        else:
            return None, f"{lhs_method}+{rhs_method}", f"symbolic: {diff}"
    except Exception as e:
        return None, None, f"simplify_error: {e}"


# Run verification on each step of each problem
results = []

for _, row in verify_df.iterrows():
    cats = categorize_problem(row["problem"])
    primary_cat = cats[0]

    for step in row["reasoning_steps"]:
        if step["type"] == "code":
            # Code blocks: we could execute them, but that's a separate verifier
            results.append({
                "problem_id": row["problem_id"], "category": primary_cat,
                "step_num": step["step_number"], "step_type": step["type"],
                "verifiable": None, "method": "code_execution",
                "detail": "requires_execution",
            })
            continue

        if step["type"] == "output":
            results.append({
                "problem_id": row["problem_id"], "category": primary_cat,
                "step_num": step["step_number"], "step_type": step["type"],
                "verifiable": None, "method": None,
                "detail": "execution_output",
            })
            continue

        # Text or answer steps: extract equations and try to verify
        equations = extract_equations(step["content"])
        if not equations:
            results.append({
                "problem_id": row["problem_id"], "category": primary_cat,
                "step_num": step["step_number"], "step_type": step["type"],
                "verifiable": None, "method": None,
                "detail": "no_equations_found",
            })
            continue

        for eq in equations:
            success, method, detail = try_verify_equation(eq)
            results.append({
                "problem_id": row["problem_id"], "category": primary_cat,
                "step_num": step["step_number"], "step_type": step["type"],
                "verifiable": success, "method": method,
                "detail": detail,
            })

res_df = pd.DataFrame(results)
print(f"Total step/expression checks: {len(res_df)}")
print(f"\n=== Verification Outcomes ===")
print(res_df["detail"].value_counts().head(15).to_string())

In [ ]:
# === Success rate by category ===
# Only count steps where we attempted equation verification (exclude code/output/no_equations)
attempted = res_df[res_df["verifiable"].notna()].copy()
total_attempted = len(attempted)
total_success = (attempted["verifiable"] == True).sum()
total_fail = (attempted["verifiable"] == False).sum()

print(f"=== SymPy Verification Success Rate ===")
print(f"Attempted: {total_attempted}")
print(f"Verified correct:  {total_success} ({total_success/max(total_attempted,1):.1%})")
print(f"Verified wrong:    {total_fail} ({total_fail/max(total_attempted,1):.1%})")
print(f"Inconclusive:      {total_attempted - total_success - total_fail}")

print(f"\n=== By Category ===")
for cat in ["algebra", "number_theory", "geometry", "combinatorics", "other"]:
    cat_data = attempted[attempted["category"] == cat]
    if len(cat_data) == 0:
        continue
    n_ok = (cat_data["verifiable"] == True).sum()
    n_fail = (cat_data["verifiable"] == False).sum()
    n_total = len(cat_data)
    print(f"  {cat:<16} {n_ok}/{n_total} verified ({n_ok/n_total:.0%})  |  {n_fail} contradictions")

# === Failure mode analysis ===
print(f"\n=== Failure Modes (parse failures) ===")
parse_fails = res_df[res_df["detail"].str.startswith("parse_fail", na=False)]
print(f"Total parse failures: {len(parse_fails)}")
for _, pf in parse_fails.head(10).iterrows():
    print(f"  [{pf['problem_id']}] step {pf['step_num']}: {pf['detail'][:80]}")

# === Step coverage: what fraction of steps have ANY verifiable content? ===
step_level = res_df.groupby(["problem_id", "step_num"]).agg(
    has_verifiable=("verifiable", lambda x: x.notna().any()),
    any_success=("verifiable", lambda x: (x == True).any()),
).reset_index()

total_steps = len(step_level)
steps_with_content = step_level["has_verifiable"].sum()
steps_verified = step_level["any_success"].sum()

print(f"\n=== Step-Level Coverage ===")
print(f"Total steps across {VERIFY_N} problems: {total_steps}")
print(f"Steps with verifiable equations: {steps_with_content} ({steps_with_content/total_steps:.0%})")
print(f"Steps successfully verified:     {steps_verified} ({steps_verified/total_steps:.0%})")

# Decision point
overall_rate = steps_verified / max(total_steps, 1)
print(f"\n{'='*60}")
if overall_rate >= 0.6:
    print(f"PASS: {overall_rate:.0%} step verification rate — SymPy is viable")
elif overall_rate >= 0.3:
    print(f"MARGINAL: {overall_rate:.0%} — consider adding Z3 or focusing on algebra/number_theory")
else:
    print(f"FAIL: {overall_rate:.0%} — SymPy alone is insufficient, need Z3 or code execution verifier")
print(f"{'='*60}")